# Rolling Walk-Forward (Experiment 3) -- EDA

Visualises `run_rolling_walk_forward_experiment`
(`thesis.system_eval.rolling_walk_forward`): the timeline is walked one
window at a time, mining+fitting from scratch on the *full* window `Wi` and
evaluating on `W(i+1)`, then discarding and repeating -- the "always
retrain" anchor, contrasted against `01_temporal_decay.ipynb`'s frozen-model
decay curve (Experiment 2).

**A structural caveat that matters for every comparison below.** Experiment
2 (in `baseline_split` mode) walks only the *post-split* tail -- `W_src` is
everything before the CSCAS baseline's own train/test boundary, held out of
the walk entirely. Experiment 3 has no such split: it walks the *whole*
timeline from t=0. At a given granularity the two happen to produce the
**same window count** (verified: 21 windows at gran 0.1, 11... wait, see
below), but window *i* in one experiment is **not** the same calendar
window as window *i* in the other. So every plot that puts both experiments
on one axis here uses **fraction of the way through each experiment's own
walk** (0 = the first evaluated window, 1 = the last) rather than the raw
window index -- the only axis the two walks genuinely share.

Sections:
- **§1** detection quality vs. step, one line per granularity (same
  panel-per-model layout as Experiment 2's decay plot), for AUC / F1 / FPR.
- **§2** the walk-forward summary table + bar chart (mean ± std per config
  -- the headline "how good is this in general" number).
- **§3** the always-retrain curve overlaid directly on Experiment 2's
  frozen-decay curve, with a shaded ±1 std band around the always-retrain
  line showing the step-to-step spread that persists even under constant
  retraining.

In [ ]:
from __future__ import annotations

%matplotlib inline
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thesis.experiments._shared import METRIC_COLS
from thesis.paths import RESULTS_DIR, ROOT

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## Config

Edit and re-run -- nothing past the loader cell needs changing.

In [ ]:
SCENARIO = "cscas"
RUN_ID = "20260921_142835"  # pinned: latest run (RUN_ID=None) is 20260921_214259,
                             # a partial symbolic/rf-only smoke test of the
                             # window_schema_cache concurrency fix -- this is the
                             # first FULL-grid run after that fix (symbolic/xgboost
                             # now present: 30/30 rows, matching every other schema)
GRANULARITY = None       # None -> finest available

# Experiment 2 (frozen decay) run to compare against in §3. None -> latest
# temporal-decay run for the same SCENARIO. Set COMPARE_TO_TEMPORAL_DECAY =
# False to skip §3 entirely (e.g. no matching Exp2 run available).
COMPARE_TO_TEMPORAL_DECAY = True
TEMPORAL_DECAY_RUN_ID = "20260919_200100"  # pinned: latest Exp2 run
                             # (TEMPORAL_DECAY_RUN_ID=None) is 20260921_214012, a
                             # partial symbolic/rf-only smoke test -- this is the
                             # last FULL-grid Exp2 run, same one 01_temporal_decay.ipynb
                             # is pinned to

PRIMARY_METRIC = "f1"    # §1/§3 default metric when not explicitly looped
MINING_SETTING = None    # which symbolic-family mining setting to use where
                         # one must be picked -- None -> first available

IMG_DIR = ROOT / "src/thesis/notebooks/experiments/img/exp2"
SAVE_FIGS = True
SAVE_DPI = 150

# Same schema vocabulary/colours as 01_temporal_decay.ipynb, kept in sync so
# figures from the two notebooks read as one system.
FEATURE_SET_ORDER = ["baseline", "symbolic", "cscas_full", "cscas_full_symbolic"]
FEATURE_SET_COLOR = {"baseline": "#AA3377", "symbolic": "#004488",
                     "cscas_full": "#DDAA33", "cscas_full_symbolic": "#228833"}
FEATURE_SET_LABEL = {"baseline": "base", "symbolic": "base_symbolic",
                     "cscas_full": "full", "cscas_full_symbolic": "full_symbolic"}
MODEL_ORDER = ["logreg", "xgboost", "rf", "iforest", "ocsvm"]
MODEL_COLOR = dict(zip(MODEL_ORDER, ["#004488", "#66CCEE", "#CCBB44", "#228833", "#EE6677"]))

# §3 colours: which *experiment* a line belongs to, not which schema/model --
# schema/model are already fixed per panel/call there.
EXP2_COLOR = "#CC3311"  # frozen schema (never retrain)
EXP3_COLOR = "#004488"  # always retrain

In [ ]:
CONFIG_COLS = ["feature_set", "mining_setting", "granularity", "model"]

sweep_dir = RESULTS_DIR / "sys-eval" / "rolling-walk-forward" / SCENARIO
run_dirs = sorted(p for p in sweep_dir.iterdir() if p.is_dir())
run_dir = (sweep_dir / RUN_ID) if RUN_ID else run_dirs[-1]


def _read_csv_maybe_empty(path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


per_step = pd.read_csv(run_dir / "per_step_results.csv", low_memory=False)
walk_forward_summary = pd.read_csv(run_dir / "walk_forward_summary.csv")
explanations = _read_csv_maybe_empty(run_dir / "explanations.csv")
lime_fidelity = _read_csv_maybe_empty(run_dir / "lime_fidelity.csv")
HAS_EXPLANATIONS = not explanations.empty

print(f"Exp3 run: {run_dir.name}")
print(f"  per_step_results   : {len(per_step):>6} rows")
print(f"  walk_forward_summary: {len(walk_forward_summary):>6} rows")
print(f"  explanations        : {len(explanations):>6} rows")
print(f"  feature_sets : {sorted(per_step['feature_set'].unique())}")
print(f"  models       : {sorted(per_step['model'].unique())}")
print(f"  granularities: {sorted(per_step['granularity'].unique())}")
print(f"  mining setts : {sorted(per_step['mining_setting'].dropna().unique())}")

# Experiment 2 (frozen decay), for §3 -- graceful no-op if unavailable.
HAS_TEMPORAL_DECAY = False
per_horizon = decay_summary = pd.DataFrame()
if COMPARE_TO_TEMPORAL_DECAY:
    td_dir = RESULTS_DIR / "sys-eval" / "temporal-decay" / SCENARIO
    td_run_dirs = sorted(p for p in td_dir.iterdir() if p.is_dir()) if td_dir.exists() else []
    if td_run_dirs:
        td_run_dir = (td_dir / TEMPORAL_DECAY_RUN_ID) if TEMPORAL_DECAY_RUN_ID else td_run_dirs[-1]
        if (td_run_dir / "per_horizon_results.csv").exists():
            per_horizon = pd.read_csv(td_run_dir / "per_horizon_results.csv", low_memory=False)
            decay_summary = pd.read_csv(td_run_dir / "decay_summary.csv")
            HAS_TEMPORAL_DECAY = True
            print(f"\nExp2 run (for §3): {td_run_dir.name}")
            print(f"  per_horizon_results: {len(per_horizon):>6} rows")
        else:
            print(f"\n[warn] no per_horizon_results.csv in {td_run_dir} -- §3 will be skipped")
    else:
        print(f"\n[warn] no temporal-decay runs found at {td_dir} -- §3 will be skipped")
else:
    print("\nCOMPARE_TO_TEMPORAL_DECAY is False -- §3 will be skipped")

## Shared helpers

In [ ]:
IMG_DIR.mkdir(parents=True, exist_ok=True)


def _slugify(*parts) -> str:
    bits = []
    for p in parts:
        if p is None or p == "":
            continue
        s = re.sub(r"[^A-Za-z0-9_.=-]+", "_", str(p)).strip("_")
        if s:
            bits.append(s)
    return "_".join(bits)


def save_fig(fig, *name_parts) -> Path | None:
    """Save `fig` to IMG_DIR as <SCENARIO>_<name_parts...>.png. No-op if
    SAVE_FIGS is False."""
    if not SAVE_FIGS:
        return None
    path = IMG_DIR / f"{_slugify(SCENARIO, *name_parts)}.png"
    fig.savefig(path, dpi=SAVE_DPI, bbox_inches="tight")
    print(f"  saved -> {path.relative_to(ROOT)}")
    return path


def fs_label(fs) -> str:
    return FEATURE_SET_LABEL.get(fs, fs)


def ms_short(ms) -> str:
    if ms is None or (isinstance(ms, float) and np.isnan(ms)):
        return ""
    ms = str(ms)
    return ms[4:].replace("_", "/") if ms.startswith("gr3_") else ms.replace("_", "/")


def resolve_gran() -> float:
    grans = sorted(per_step["granularity"].unique())
    if GRANULARITY is None:
        return grans[0]
    if GRANULARITY not in grans:
        raise ValueError(f"GRANULARITY={GRANULARITY} not in run ({grans})")
    return GRANULARITY


GRAN = resolve_gran()
ALL_GRANULARITIES = sorted(per_step["granularity"].unique())
FEATURE_SETS = [fs for fs in FEATURE_SET_ORDER if fs in set(per_step["feature_set"])]
MODELS = [m for m in MODEL_ORDER if m in set(per_step["model"])]
_unknown_models = sorted(set(per_step["model"]) - set(MODEL_ORDER))
if _unknown_models:
    _extra_pal = plt.get_cmap("tab10")
    for _i, _m in enumerate(_unknown_models):
        MODEL_COLOR[_m] = _extra_pal(_i % 10)
    MODELS += _unknown_models
MINING_SETTINGS = sorted(per_step.loc[per_step["feature_set"] == "symbolic", "mining_setting"].dropna().unique())
MS_ONE = MINING_SETTING or (MINING_SETTINGS[0] if MINING_SETTINGS else None)

# step_i runs 0 .. n_windows-2 (evaluating W(i+1) for i up to the second-to-
# last window) -- step_fraction puts it on the same [0, 1] axis as Exp2's
# native horizon_fraction column, so different granularities (different
# step counts) and different experiments (different walks) can share an
# x-axis. See the title markdown for why raw index isn't safe to share
# across experiments.
per_step["step_fraction"] = per_step["step_i"] / (per_step["n_windows"] - 2)

_gran_pal = plt.get_cmap("plasma")
GRAN_COLOR = dict(zip(
    ALL_GRANULARITIES,
    [_gran_pal(x) for x in np.linspace(0.15, 0.80, max(1, len(ALL_GRANULARITIES)))],
))


def _ms_mask(df: pd.DataFrame, feature_set: str, mining_setting) -> pd.Series:
    """feature_set-and-mining_setting mask, NaN-safe (baseline/cscas_full
    carry no mining_setting -- a plain `==` never matches NaN)."""
    m = df["feature_set"].eq(feature_set)
    if feature_set in ("symbolic", "cscas_full_symbolic"):
        ms = mining_setting or MS_ONE
        m &= df["mining_setting"].eq(ms)
    else:
        m &= df["mining_setting"].isna()
    return m

## 1. Detection quality vs. step, one line per granularity

Same panel layout as Experiment 2's decay plot (one panel per model) -- but
here each **line is a granularity** rather than a schema, since the
question is "does the always-retrain curve look different at a finer vs.
coarser walk", not a schema comparison. `feature_set` is fixed per call
(default `symbolic` if present). x-axis is `step_fraction` (0 = the first
evaluated step, 1 = the last), so gran 0.1's 10 steps and gran 0.05's 20
steps land on the same axis. The shaded band is the IQR across mining
settings at that step (a single setting in this run, so it collapses to the
line -- see `01_temporal_decay.ipynb`'s note on the same point).

In [ ]:
def _step_series(feature_set: str, model: str, metric: str, gran: float, mining_setting=None):
    m = (_ms_mask(per_step, feature_set, mining_setting) & per_step["model"].eq(model)
         & per_step["granularity"].eq(gran))
    sub = per_step[m]
    if sub.empty or metric not in sub:
        return None
    g = sub.groupby("step_fraction")[metric]
    return g.median().index.values, g.median().values, g.quantile(0.25).values, g.quantile(0.75).values


def plot_step_metric_by_granularity(metric: str = None, feature_set: str = None, models=None) -> None:
    metric = metric or PRIMARY_METRIC
    feature_set = feature_set or ("symbolic" if "symbolic" in FEATURE_SETS else FEATURE_SETS[0])
    models = models or MODELS
    ncols = min(2, len(models))
    nrows = -(-len(models) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 4.2 * nrows), squeeze=False)
    for ax, model in zip(axes.flat, models):
        any_data = False
        for gran in ALL_GRANULARITIES:
            s = _step_series(feature_set, model, metric, gran)
            if s is None:
                continue
            any_data = True
            h, med, q25, q75 = s
            color = GRAN_COLOR[gran]
            ax.plot(h, med, "-o", ms=4, lw=1.8, color=color, label=f"gran={gran:g}", zorder=3)
            if (q75 > q25).any():
                ax.fill_between(h, q25, q75, color=color, alpha=0.15, lw=0)
        if not any_data:
            ax.text(0.5, 0.5, "no data for this config", ha="center", va="center",
                    transform=ax.transAxes, fontsize=9, color="grey")
        ax.set_title(model)
        ax.set_xlabel("fraction of walk (0=first step, 1=last)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
        ax.margins(x=0.02)
    for ax in axes.flat[len(models):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: always-retrain {metric} vs step -- {fs_label(feature_set)}")
    fig.tight_layout()
    save_fig(fig, "sec1", "step_metric", fs_label(feature_set), metric)


plot_step_metric_by_granularity("auc")
plot_step_metric_by_granularity("f1")
plot_step_metric_by_granularity("fpr")

### All metrics, one line per schema -- the full picture for a single detector

The Exp3 analogue of `01_temporal_decay.ipynb`'s `plot_all_metrics`: a 2x3
grid of every metric, one line per `feature_set` (schema), fixed model and
granularity, with the window's alert-group composition (grey=benign,
red=attack) as bars on a twin axis behind the lines. x-axis is
`step_fraction` (0=first step, 1=last), same convention as §1 above. Only
`show_groups="total"` is available here (every group in that step's window
Wi) -- Exp3 has no novelty tracking the way Exp2's W_src-relative "novel"
mode does, since there's no fixed source window to compare against.

In [ ]:
def group_composition_series(gran: float = None) -> pd.DataFrame:
    """Per-step (benign, attack) alert-group counts at `gran` (default: the
    global GRAN) -- identical across every config at that granularity."""
    gran = GRAN if gran is None else gran
    sub = per_step[per_step["granularity"].eq(gran)]
    if sub.empty:
        return pd.DataFrame()
    cols = ["step_fraction", "n_alert_groups", "n_attack"]
    out = (sub[cols].drop_duplicates("step_fraction")
           .set_index("step_fraction").sort_index())
    out["n_benign"] = out["n_alert_groups"] - out["n_attack"]
    return out[["n_benign", "n_attack"]]


def _overlay_group_bars(ax, gran: float = None) -> None:
    """Stacked benign/attack alert-group counts for each step's window, on a
    twin right-hand axis rendered behind the metric lines. `gran` (default:
    the global GRAN) must match whatever gran the caller's metric lines are
    plotting, or the bars describe a different walk than the lines do."""
    groups = group_composition_series(gran)
    if groups.empty:
        return
    steps = groups.index.values
    gap = np.median(np.diff(np.sort(steps))) if len(steps) > 1 else 0.05
    width = 0.8 * gap
    ax2 = ax.twinx()
    ax2.bar(steps, groups["n_benign"], color="#999999", alpha=0.35,
            width=width, zorder=1, label="benign groups")
    ax2.bar(steps, groups["n_attack"], bottom=groups["n_benign"],
            color="#9E2A2B", alpha=0.55, width=width, zorder=1, label="attack groups")
    ax2.set_ylabel("alert groups\n(grey=benign, red=attack)", color="#555555", fontsize=7)
    ax2.tick_params(axis="y", colors="#555555", labelsize=6)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)


def plot_all_metrics(model: str = None, gran: float = None,
                     show_groups: str | None = "total") -> None:
    """gran: which granularity to plot (default: the global GRAN)."""
    model = model or MODELS[0]
    gran = GRAN if gran is None else gran
    metrics = [PRIMARY_METRIC] + [m for m in METRIC_COLS if m != PRIMARY_METRIC]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), squeeze=False)
    for ax, metric in zip(axes.flat, metrics):
        if show_groups:
            _overlay_group_bars(ax, gran=gran)
        for fs in FEATURE_SETS:
            s = _step_series(fs, model, metric, gran)
            if s is None:
                continue
            h, med, q25, q75 = s
            ax.plot(h, med, "-o", ms=3.5, lw=1.6, color=FEATURE_SET_COLOR[fs], label=fs_label(fs))
            if (q75 > q25).any():
                ax.fill_between(h, q25, q75, color=FEATURE_SET_COLOR[fs], alpha=0.18, lw=0)
            ax.scatter(h[:1], med[:1], color=FEATURE_SET_COLOR[fs], marker="*", s=110,
                       edgecolor="black", linewidth=0.5, zorder=5)
        ax.set_title(metric)
        ax.set_xlabel("fraction of walk (0=first step, 1=last)")
        ax.grid(alpha=0.25)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"{SCENARIO}: always-retrain, all metrics vs step -- model={model}, gran={gran:g}")
    fig.tight_layout()
    save_fig(fig, "sec1", "allmetrics", model, f"gran={gran:g}")


plot_all_metrics("rf")
plot_all_metrics("xgboost")
plot_all_metrics("rf", 0.1)
plot_all_metrics("xgboost", 0.1)

## 2. Walk-forward summary (headline number)

Mean and std of `auc`, `f1`, and `fpr` across every step actually run, one
row per config -- the "how good is this config in general use" number this
experiment exists to produce.

In [ ]:
sort_col = f"{PRIMARY_METRIC}_mean"
wfs_view = walk_forward_summary.copy()
wfs_view["schema"] = wfs_view["feature_set"].map(fs_label)
summary_sorted = (
    wfs_view.sort_values(sort_col, ascending=False) if sort_col in wfs_view else wfs_view
)
summary_sorted

In [ ]:
def config_label(row) -> str:
    ms = f"/{ms_short(row['mining_setting'])}" if pd.notna(row["mining_setting"]) else ""
    return f"{fs_label(row['feature_set'])}{ms} {row['model']} g={row['granularity']:g}"


mean_col, std_col = f"{PRIMARY_METRIC}_mean", f"{PRIMARY_METRIC}_std"
plot_order = summary_sorted.sort_values(["granularity", "feature_set", "mining_setting", "model"]).reset_index(drop=True)
labels = plot_order.apply(config_label, axis=1)
colors = [FEATURE_SET_COLOR[fs] for fs in plot_order["feature_set"]]

fig, ax = plt.subplots(figsize=(max(6, 0.55 * len(plot_order)), 4.5))
ax.bar(labels, plot_order[mean_col], yerr=plot_order[std_col], color=colors, capsize=3)
ax.set_ylabel(f"{PRIMARY_METRIC} (mean +/- std across steps)")
ax.set_title(f"{SCENARIO}: walk-forward {PRIMARY_METRIC} by config")
plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=7)
fig.tight_layout()
save_fig(fig, "sec2", "walk_forward_summary", PRIMARY_METRIC)

## 3. Frozen decay vs. always-retrain, overlaid

Experiment 2's frozen-schema decay curve and Experiment 3's always-retrain
curve, **on the same axes** (fraction of the way through each experiment's
own walk -- see the title markdown for why raw step/horizon index isn't
shared between them), for a fixed schema and granularity, one panel per
model. The shaded band around the always-retrain line is its mean ± 1 std
across every step actually run (from `walk_forward_summary` -- the
"steady-state" spread that persists even though every step retrains from
scratch), so a reader can see whether the frozen curve's decay ever crosses
into that band, not just where the two curves' endpoints land.

In [ ]:
def _horizon_series(feature_set: str, model: str, metric: str, gran: float, mining_setting=None):
    if not HAS_TEMPORAL_DECAY:
        return None
    m = (_ms_mask(per_horizon, feature_set, mining_setting) & per_horizon["model"].eq(model)
         & per_horizon["granularity"].eq(gran))
    sub = per_horizon[m]
    if sub.empty or metric not in sub or "horizon_fraction" not in sub:
        return None
    g = sub.groupby("horizon_fraction")[metric]
    return g.median().index.values, g.median().values


def _wfs_mean_std(feature_set: str, model: str, metric: str, gran: float, mining_setting=None):
    m = (_ms_mask(walk_forward_summary, feature_set, mining_setting) & walk_forward_summary["model"].eq(model)
         & walk_forward_summary["granularity"].eq(gran))
    sub = walk_forward_summary[m]
    mcol, scol = f"{metric}_mean", f"{metric}_std"
    if sub.empty or mcol not in sub:
        return None
    row = sub.iloc[0]
    return float(row[mcol]), (float(row[scol]) if pd.notna(row.get(scol)) else 0.0)


def plot_frozen_vs_retrain(metric: str = None, feature_set: str = None, granularity: float = None,
                           models=None, show_band: bool = True) -> None:
    if not HAS_TEMPORAL_DECAY:
        print("no matching temporal-decay run loaded (COMPARE_TO_TEMPORAL_DECAY=False, or none found) -- skipping")
        return
    metric = metric or PRIMARY_METRIC
    feature_set = feature_set or ("symbolic" if "symbolic" in FEATURE_SETS else FEATURE_SETS[0])
    granularity = granularity or GRAN
    models = models or MODELS
    fig, axes = plt.subplots(1, len(models), figsize=(5.6 * len(models), 4.4), squeeze=False, sharey=False)
    for ax, model in zip(axes[0], models):
        e2 = _horizon_series(feature_set, model, metric, granularity)
        if e2 is not None:
            ax.plot(e2[0], e2[1], "-o", ms=4, lw=1.8, color=EXP2_COLOR,
                    label="Exp2: frozen schema", zorder=3)
        else:
            ax.text(0.5, 0.7, "no Exp2 data for this config", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color=EXP2_COLOR)
        e3 = _step_series(feature_set, model, metric, granularity)
        if e3 is not None:
            h3, med3, _, _ = e3
            ax.plot(h3, med3, "-o", ms=4, lw=1.8, color=EXP3_COLOR,
                    label="Exp3: always retrain", zorder=3)
        else:
            ax.text(0.5, 0.3, "no Exp3 data for this config", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color=EXP3_COLOR)
        if show_band:
            ms = _wfs_mean_std(feature_set, model, metric, granularity)
            if ms is not None:
                mean_v, std_v = ms
                ax.axhspan(mean_v - std_v, mean_v + std_v, color=EXP3_COLOR, alpha=0.15, zorder=1,
                          label="Exp3 +/-1 std (step-to-step spread)")
                ax.axhline(mean_v, color=EXP3_COLOR, lw=1, ls=":", zorder=2)
        ax.set_title(model)
        ax.set_xlabel("fraction of walk (0=first, 1=last)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    fig.suptitle(f"{SCENARIO}: frozen decay vs. always-retrain -- "
                f"{fs_label(feature_set)}, gran={granularity:g} ({metric})")
    fig.tight_layout()
    save_fig(fig, "sec3", "frozen_vs_retrain", fs_label(feature_set), metric, f"gran={granularity:g}")


plot_frozen_vs_retrain("auc")
plot_frozen_vs_retrain("f1")
plot_frozen_vs_retrain("fpr")

In [ ]:
# Numeric companion to the plot above: Exp2's h=0 (best case, before decay)
# and h=h_max (frozen, fully decayed) vs. Exp3's steady-state mean, per
# config -- the always-retrain gain a perfect retraining policy buys over
# never retraining, the number Experiment 4's drift monitor is compared
# against.
comparison = None
if HAS_TEMPORAL_DECAY and not decay_summary.empty:
    ds = decay_summary.copy()
    ds[f"{PRIMARY_METRIC}_at_h_max"] = ds.apply(
        lambda r: r.get(f"{PRIMARY_METRIC}_at_h{int(r['h_max'])}", np.nan), axis=1
    )
    comparison = walk_forward_summary.merge(
        ds[CONFIG_COLS + [f"{PRIMARY_METRIC}_at_h0", f"{PRIMARY_METRIC}_at_h_max"]],
        on=CONFIG_COLS, how="inner",
    )
    comparison["always_retrain_gain"] = (
        comparison[f"{PRIMARY_METRIC}_mean"] - comparison[f"{PRIMARY_METRIC}_at_h_max"]
    )
    comparison["schema"] = comparison["feature_set"].map(fs_label)
    print(f"{len(comparison)}/{len(walk_forward_summary)} configs matched by "
          "(feature_set, mining_setting, granularity, model)")
else:
    print("no comparison table -- §3's plot above already covers the same ground visually")
comparison

### F1-only small multiple: one figure per (model, granularity), all schemas

Compact companion to the model-per-panel view above -- here **schema is the
panel** (2x2: base / base_symbolic / full / full_symbolic) and model +
granularity are fixed per figure, F1 only. Same shared x-axis (fraction of
the way through each experiment's own walk) and colours (Exp2 red, Exp3
blue) as above. One figure per (model, granularity) -- 4 total.

In [ ]:
def plot_frozen_vs_retrain_grid(model: str, gran: float, metric: str = "f1",
                                feature_sets=None) -> None:
    feature_sets = feature_sets or FEATURE_SETS
    ncols = 2
    nrows = -(-len(feature_sets) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.0 * nrows), squeeze=False)
    for ax, fs in zip(axes.flat, feature_sets):
        e2 = _horizon_series(fs, model, metric, gran)
        if e2 is not None:
            ax.plot(e2[0], e2[1], "-o", ms=3.5, lw=1.6, color=EXP2_COLOR,
                    label="Exp2: frozen schema", zorder=3)
        else:
            ax.text(0.5, 0.7, "no Exp2 data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color=EXP2_COLOR)
        e3 = _step_series(fs, model, metric, gran)
        if e3 is not None:
            h3, med3, _, _ = e3
            ax.plot(h3, med3, "-o", ms=3.5, lw=1.6, color=EXP3_COLOR,
                    label="Exp3: always retrain", zorder=3)
        else:
            ax.text(0.5, 0.3, "no Exp3 data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color=EXP3_COLOR)
        ax.set_title(fs_label(fs))
        ax.set_xlabel("fraction of walk (0=first, 1=last)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    for ax in axes.flat[len(feature_sets):]:
        ax.set_visible(False)
    fig.suptitle(f"{SCENARIO}: frozen decay vs. always-retrain -- "
                f"model={model}, gran={gran:g} ({metric})")
    fig.tight_layout()
    save_fig(fig, "sec3", "frozen_vs_retrain_grid", model, metric, f"gran={gran:g}")


for _model in MODELS:
    for _gran in ALL_GRANULARITIES:
        plot_frozen_vs_retrain_grid(_model, _gran)

## 4. Headline tables: volatility and ceiling comparison

Two LaTeX tables for the thesis text:

- **Table A (volatility)** -- does performance still swing step-to-step even
  though every step gets a freshly mined+fit model? Same 5-column shape as
  the Frozen-Schema decay-rate table (Table 1), but computed over Exp3's
  own step series (F1, in step order) instead of Exp2's horizon series. One
  table per model (RF in the main text, XGBoost in the appendix), rows =
  schema x granularity.
- **Table B (ceiling comparison)** -- direct number: does constant
  retraining actually beat the frozen model, even at the frozen model's
  best moment (h=0, before any decay)? Rows = schema x model, one table per
  granularity (0.05 main text, 0.1 appendix).

Schema definitions (used in both captions): **Base** = the reduced core
feature set (no mining); **Base + symbolic** = Base plus the mined
symbolic/contrast-tree predicates; **Full** = the CSCAS paper's own full
reference feature set (non-deployable oracle columns included); **Full +
symbolic** = Full plus the mined symbolic predicates.

In [ ]:
def _fmt(v, nd=3) -> str:
    return "--" if pd.isna(v) else f"{v:.{nd}f}"


def _step_deltas(series_by_step: pd.Series) -> dict | None:
    """5-column volatility summary over a step-ordered series -- same
    formulas as Table 1's decay-rate columns, applied to Exp3's own step
    series instead of Exp2's horizon series."""
    vals = series_by_step.sort_index().values
    if len(vals) < 2:
        return None
    diffs = np.diff(vals)
    return {
        "delta_step0_step1": vals[1] - vals[0],
        "mean_abs_delta": float(np.mean(np.abs(diffs))),
        "max_drop": float(max(0.0, np.max(-diffs))),
        "max_rise": float(max(0.0, np.max(diffs))),
        "net_drift": vals[-1] - vals[0],
        "n": len(vals),
    }


def build_volatility_table(model: str, metric: str = None) -> pd.DataFrame:
    metric = metric or PRIMARY_METRIC
    rows = []
    for gran in ALL_GRANULARITIES:
        for fs in FEATURE_SETS:
            m = (_ms_mask(per_step, fs, None) & per_step["model"].eq(model)
                 & per_step["granularity"].eq(gran))
            sub = per_step[m]
            if sub.empty or metric not in sub:
                continue
            d = _step_deltas(sub.set_index("step_i")[metric])
            if d is None:
                continue
            rows.append({"granularity": gran, "schema": fs_label(fs), **d})
    return pd.DataFrame(rows)


def volatility_table_to_latex(df: pd.DataFrame, model: str, metric_name: str = "F1") -> str:
    lines = [
        r"\begin{table}[h]", r"\centering", r"\footnotesize",
        r"\begin{tabular}{llrrrrrr}", r"\toprule",
        r"\textbf{Gran} & \textbf{Schema} & $n$ & $\Delta_{(0\to1)}$ & "
        r"Mean $|\Delta|$ & Max drop & Max rise & Net drift \\",
        r"\midrule",
    ]
    for gran, g in df.groupby("granularity"):
        g = g.reset_index(drop=True)
        for i, row in g.iterrows():
            gran_cell = f"{gran:g}" if i == 0 else ""
            lines.append(
                f"{gran_cell} & {row['schema']} & {int(row['n'])} & "
                f"{_fmt(row['delta_step0_step1'])} & {_fmt(row['mean_abs_delta'])} & "
                f"{_fmt(row['max_drop'])} & {_fmt(row['max_rise'])} & {_fmt(row['net_drift'])} \\\\"
            )
        lines.append(r"\midrule")
    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular}")
    caption = (
        f"Always-retrain {metric_name} step-to-step volatility, {model}. "
        r"$\Delta_{(0\to1)} = s(\text{step}_1) - s(\text{step}_0)$; "
        r"Mean $|\Delta|$ = mean absolute change between consecutive steps; "
        r"Max drop/rise = largest single-step fall/gain; Net drift = "
        r"$s(\text{last}) - s(\text{step}_0)$, for this run's mining setting "
        f"({ms_short(MS_ONE)}). "
        "Unlike Table~\\ref{tab:decay-rate}, where Mean $|\\Delta|$ tracks one "
        "frozen model's drift against a changing environment, here \\textbf{every "
        "step re-mines and re-fits a new model from scratch} -- so Mean $|\\Delta|$ "
        "reflects environment drift and model-refit variance combined, and is not "
        "directly comparable to the frozen-schema table's column of the same name."
    )
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{tab:always-retrain-volatility-" + model + "}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


vol_rf = build_volatility_table("rf")
print(volatility_table_to_latex(vol_rf, "RF"))
vol_rf

### Table A (XGBoost) -- appendix

In [ ]:
vol_xgb = build_volatility_table("xgboost")
print(volatility_table_to_latex(vol_xgb, "XGBoost"))
vol_xgb

### Table B: frozen vs. always-retrain ceiling comparison

In [ ]:
def build_ceiling_table(models, gran: float) -> pd.DataFrame:
    metric = PRIMARY_METRIC
    h0_col = f"{metric}_at_h0"
    mean_col, std_col = f"{metric}_mean", f"{metric}_std"
    rows = []
    for fs in FEATURE_SETS:
        for model in models:
            m_ds = (_ms_mask(decay_summary, fs, None) & decay_summary["model"].eq(model)
                    & decay_summary["granularity"].eq(gran))
            m_wfs = (_ms_mask(walk_forward_summary, fs, None) & walk_forward_summary["model"].eq(model)
                     & walk_forward_summary["granularity"].eq(gran))
            sub_ds = decay_summary[m_ds]
            sub_wfs = walk_forward_summary[m_wfs]
            if sub_ds.empty or sub_wfs.empty or h0_col not in sub_ds:
                continue
            frozen_h0 = float(sub_ds.iloc[0][h0_col])
            ar_mean = float(sub_wfs.iloc[0][mean_col])
            ar_std = float(sub_wfs.iloc[0][std_col]) if pd.notna(sub_wfs.iloc[0].get(std_col)) else np.nan
            n_steps = int(sub_wfs.iloc[0]["n_steps"])
            rows.append({
                "schema": fs_label(fs), "model": model,
                "frozen_h0": frozen_h0, "ar_mean": ar_mean,
                "gap": ar_mean - frozen_h0, "ar_std": ar_std, "n_steps": n_steps,
            })
    return pd.DataFrame(rows)


def ceiling_table_to_latex(df: pd.DataFrame, gran: float, metric_name: str = "F1") -> str:
    lines = [
        r"\begin{table}[h]", r"\centering", r"\footnotesize",
        r"\begin{tabular}{llrrrrr}", r"\toprule",
        r"\textbf{Schema} & \textbf{Model} & Frozen $h_0$ & Always-retrain mean & "
        r"Gap & Always-retrain std & $n$ steps \\",
        r"\midrule",
    ]
    for schema, g in df.groupby("schema", sort=False):
        g = g.reset_index(drop=True)
        for i, row in g.iterrows():
            schema_cell = row["schema"] if i == 0 else ""
            lines.append(
                f"{schema_cell} & {row['model']} & {_fmt(row['frozen_h0'])} & "
                f"{_fmt(row['ar_mean'])} & {_fmt(row['gap'])} & {_fmt(row['ar_std'])} & "
                f"{int(row['n_steps'])} \\\\"
            )
        lines.append(r"\midrule")
    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular}")
    caption = (
        f"Frozen schema vs. always-retrain, {metric_name}, gran={gran:g}. "
        r"Frozen $h_0$ = the frozen model's own best-case score (Experiment 2, "
        "before any decay); Always-retrain mean/std = Experiment 3's "
        r"walk\_forward\_summary mean $\pm$ std across every step actually run "
        "(the row of Table~\\ref{tab:always-retrain-volatility-RF} at this "
        r"schema/granularity gives the same run's step-to-step volatility); "
        "Gap = Always-retrain mean $-$ Frozen $h_0$ (positive = constant "
        "retraining beats the frozen model even at its freshest)."
    )
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{tab:frozen-vs-retrain-gran" + f"{gran:g}".replace(".", "") + "}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


ceil_05 = build_ceiling_table(MODELS, 0.05)
print(ceiling_table_to_latex(ceil_05, 0.05))
ceil_05

### Table B (gran=0.1) -- appendix

In [ ]:
ceil_01 = build_ceiling_table(MODELS, 0.1)
print(ceiling_table_to_latex(ceil_01, 0.1))
ceil_01